# Lab 4 — Clasificarea Florilor cu PyTorch (Computer Vision)

Acest notebook rezolva complet cerintele laboratorului 4 (*Bildverarbeitung mit PyTorch*) pe dataset-ul **Flowers** (`daisy`, `dandelion`, `roses`, `sunflowers`, `tulips`).

### Structura rezolvarii
1. **Configurare, Hardware si Date** — verificam PyTorch/CUDA, incarcam si exploram imaginile, aplicam preprocesarea si construim split-ul de **5-Fold Stratified Cross-Validation** cu acelasi `SEED = 42`.
2. **Arhitectura Modulara a Modelelor** — construim o retea flexibila care permite experimentarea cu 0–6 straturi convolutionale, cu/fara Pooling, 1–3 straturi ascunse (Dense/FC), mai multe functii de activare, Batch Normalization, Dropout (0–0.5) si Recurenta (GRU). Implementam si inspectarea dimensiunilor tensorilor/matricelor si numararea parametrilor.
3. **Pipeline de Antrenare & Validare** — implementam antrenarea pe 5 fold-uri cu **Early Stopping**, regularizare **L1/L2**, **Data Augmentation**, calculul metricii **Macro-F1**, salvarea modelelor pe disc si logarea automata a rezultatelor.
4. **Experimente (Grid Search Sistematic)** — rulam 18 configuratii alese sistematic pentru a izola efectul fiecarui hiperparametru asupra calitatii predictiei si a timpului de rulare.
5. **Rezultate, Grafice si Analiza Erorilor** — afisam evolutia *Train/Val Loss*, *Train/Val Macro-F1* si *Matricea de Confuzie* pentru **toate experimentele**, comparam dimensiunea pe disc a modelelor, inspectam dimensiunile vectorilor/matricelor pentru cel mai bun model si analizam imaginile clasificate gresit de majoritatea modelelor.
6. **Comparatie CPU vs GPU** — masuram timpul de antrenare pe CPU si pe GPU pentru 4 arhitecturi diferite si calculam accelerarea obtinuta.
7. **Concluzii si Interpretari** — sintetizam intuitiv cum influenteaza fiecare hiperparametru performanta si durata antrenarii.

## 1. Configurare, Verificare Hardware si Date

### 1.1 Importuri, Seed si Hardware (CPU / GPU)
Inainte de orice, fixam un **random seed** (`SEED = 42`) pentru Python, NumPy si PyTorch.

> **De ce este important seed-ul?**
> Cand comparam mai multe arhitecturi de retele neuronale, vrem sa stim ca diferenta de scor vine din arhitectura sau hiperparametri, nu din noroc (cum au fost initializate ponderile sau cum au fost impartite datele). Acelasi seed ne garanteaza ca toate experimentele folosesc exact aceleasi 5 fold-uri.

In [ ]:
import os
import gc
import json
import time
import copy
import random
import platform
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from IPython.display import display, Markdown

# Reproductibilitate
SEED = 42

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# Detectare CPU si GPU
def get_cpu_name():
    try:
        if platform.system() == "Windows":
            out = subprocess.check_output(
                ["powershell", "-Command", "(Get-CimInstance Win32_Processor).Name"],
                text=True
            ).strip()
            if out:
                return out
    except Exception:
        pass
    return platform.processor() or "Unknown CPU"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU_NAME = get_cpu_name()
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A (doar CPU)"

print(f"PyTorch:     {torch.__version__}")
print(f"Device:      {DEVICE}")
print(f"CPU Model:   {CPU_NAME}")
print(f"GPU Model:   {GPU_NAME}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU VRAM:    {vram_gb:.2f} GB")

# Cai si directoare de lucru
DATA_DIR = Path("../data/flower_photos")
SAVE_DIR = Path("saved_models")
RESULTS_DIR = Path("results")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"Nu am gasit dataset-ul la {DATA_DIR.resolve()}"

### 1.2 Explorarea Dataset-ului (EDA)
Dataset-ul **Flowers** contine fotografii reale organizate in 5 foldere, cate unul pentru fiecare specie de floare. Imaginile nu au *bounding boxes* — intreaga imagine reprezinta o floare (uneori cu fundal complex, iarba, mana, alte flori etc.).

In aceasta celula:
- scanam toate imaginile si etichetele lor;
- verificam distributia claselor (daca sunt echilibrate sau nu);
- verificam dimensiunile originale ale imaginilor si afisam cateva exemple din fiecare clasa.

In [ ]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
CLASS_NAMES = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

image_paths = []
labels = []
orig_sizes = []

for cls_name in CLASS_NAMES:
    cls_dir = DATA_DIR / cls_name
    for img_path in sorted(cls_dir.iterdir()):
        if img_path.suffix.lower() in VALID_EXTS:
            image_paths.append(img_path)
            labels.append(CLASS_TO_IDX[cls_name])

labels = np.array(labels, dtype=np.int64)

# Verificam dimensiunile originale pe un esantion de 300 imagini (rapid)
sample_idx = np.linspace(0, len(image_paths) - 1, min(300, len(image_paths)), dtype=int)
for idx in sample_idx:
    with Image.open(image_paths[idx]) as im:
        orig_sizes.append(im.size)  # (width, height)

widths, heights = zip(*orig_sizes)
print(f"Total imagini: {len(image_paths)} | Numar clase: {NUM_CLASSES} -> {CLASS_NAMES}")
print(f"Latime (px):   min={min(widths)}, mediana={int(np.median(widths))}, max={max(widths)}")
print(f"Inaltime (px): min={min(heights)}, mediana={int(np.median(heights))}, max={max(heights)}")

# Grafic distributie clase
counts = [int((labels == i).sum()) for i in range(NUM_CLASSES)]
fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.bar(CLASS_NAMES, counts, color=sns.color_palette("viridis", NUM_CLASSES))
ax.set_title("Distributia imaginilor pe clase (Flowers Dataset)")
ax.set_ylabel("Numar de imagini")
for bar, c in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, c + 12, str(c), ha="center", fontweight="bold")
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.show()

# Exemple vizuale din fiecare clasa
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(10, 10))
rng = np.random.default_rng(SEED)
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    cls_indices = np.where(labels == cls_idx)[0]
    chosen = rng.choice(cls_indices, size=4, replace=False)
    for col, img_idx in enumerate(chosen):
        ax = axes[cls_idx, col]
        with Image.open(image_paths[img_idx]) as im:
            ax.imshow(im.convert("RGB"))
        ax.axis("off")
        if col == 0:
            ax.set_title(f"{cls_name} (n={counts[cls_idx]})", fontsize=11, fontweight="bold", loc="left")
plt.suptitle("Exemple aleatorii din fiecare clasa", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 1.3 Preprocesare, Downscaling, Caching in RAM si 5-Fold Cross-Validation

> **De ce facem downscaling la `64 x 64` (sau `96 x 96`) si pastram imaginile pre-redimensionate in RAM?**
> 1. Imaginile originale au rezolutii diferite (de la ~150 px la ~500 px). Retelele neuronale au nevoie de imagini de **aceeasi dimensiune** pentru a le procesa in batch-uri.
> 2. O rezolutie de `64 x 64` pastreaza culoarea, forma petalelor si textura de baza, dar reduce masiv memoria VRAM necesara (important pe GTX 1650 cu 4 GB VRAM) si face rularea a 18 experimente x 5 fold-uri fezabila in cateva minute in loc de ore.
> 3. Citirea a 3.670 de fisiere `.jpg` de pe disc la fiecare epoca (pe Windows) este lenta. Daca redimensionam imaginile o singura data la `64 x 64` si le pastram ca `uint8` in RAM, ocupa doar ~45 MB (`3670 * 64 * 64 * 3` bytes) si antrenarea devine de **5–10x mai rapida**!

**Normalizare & Data Augmentation:**
- Normalizam pixelii in intervalul `[-1, 1]` (echivalent cu `mean=0.5, std=0.5` pe cele 3 canale RGB), ceea ce ajuta gradientii sa fie stabili.
- Pentru experimentele cu **Augmentation**, aplicam la fiecare epoca transformari aleatorii *doar pe setul de antrenare* (niciodata pe validare!): oglindire orizontala (`RandomHorizontalFlip`), rotatii usoare (`RandomRotation(20)`), decupare/zoom usor (`RandomResizedCrop`) si mici variatii de luminozitate/contrast (`ColorJitter`). Intuitiv, modelul vede versiuni usor diferite ale aceleiasi flori si invata trasaturi generale, nu sa memoreze pixelii.

**5-Fold Stratified Cross-Validation:**
Impartim dataset-ul in 5 bucati (fold-uri) pastrand proportia fiecarei specii de flori in fiecare fold (`StratifiedKFold`). La fiecare fold, 80% din imagini sunt folosite pentru antrenare si 20% pentru validare.

In [ ]:
IMG_SIZE = 64
N_FOLDS = 5
BATCH_SIZE = 64

# 1. Pre-incarcam si redimensionam toate imaginile o singura data in RAM (uint8: ~45 MB)
t0 = time.time()
all_images_np = np.empty((len(image_paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
for i, p in enumerate(image_paths):
    with Image.open(p) as im:
        im_rgb = im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        all_images_np[i] = np.asarray(im_rgb, dtype=np.uint8)
print(f"Incarcat si redimensionat {len(all_images_np)} imagini la {IMG_SIZE}x{IMG_SIZE} in {time.time() - t0:.2f}s "
      f"({all_images_np.nbytes / (1024**2):.1f} MB in RAM).")

# 2. Transformari (Train fara augmentare, Train cu augmentare, Validare)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

aug_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.04),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class CachedFlowerDataset(Dataset):
    def __init__(self, images_np, labels_np, indices, transform=None):
        self.images_np = images_np
        self.labels_np = labels_np
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        img = self.images_np[idx]
        y = int(self.labels_np[idx])
        if self.transform is not None:
            img = self.transform(img)
        return img, y

# 3. Generam cele 5 fold-uri cu acelasi SEED pentru toate experimentele
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(np.zeros(len(labels)), labels))

for f_idx, (tr_idx, va_idx) in enumerate(FOLDS):
    print(f"Fold {f_idx + 1}: train={len(tr_idx)} imagini, val={len(va_idx)} imagini")